# Sample QC

This performs QC on the individuals from the HCM_GWAS_START Cohort individuals.

In [ ]:
%%capture
## Python Package Import
import sys
import os 
import numpy as np
import pandas as pd
from datetime import datetime

##Ensuring dsub is up to date
!pip3 install --upgrade dsub

#Defining environment variables
# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env JOB_ID={LINE_COUNT_JOB_ID}
## Defining necessary pathways
my_bucket = os.environ['WORKSPACE_BUCKET']
project_name='HCM_GWAS_V2'
## Setting for running dsub jobs
pd.set_option('display.max_colwidth', 0)

USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env USER_NAME={USER_NAME}
%env PROJECT_NAME=HCM_GWAS_V2

## ID of High Quality, Common, Independent Variants for Sample QC

This follows the Sealock et al, 2025 approach of first identifying the high-quality variants AND then running the sample QC.
This is done for both the

1) Array data
2) srWGS ACAF data

In [ ]:
%%capture
JOB_NAME='SAMPLE_QC'

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env JOB_NAME={JOB_NAME}
%env PHENO_FILEPATH={PHENO_FILEPATH}
%env COV_FILEPATH={COV_FILEPATH}

## Analysis Results Folder 
line_count_results_folder = os.path.join(
    os.getenv('WORKSPACE_BUCKET'),
    'dsub',
    'results',
    os.getenv('PROJECT_NAME'),
    JOB_NAME,
    datetime.now().strftime('%Y%m%d'))

## Where the output files will go
output_files = os.path.join(line_count_results_folder)

OUTPUT_FILES = output_files

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env OUTPUT_FILES={OUTPUT_FILES}

### HQ Array Variants

In [ ]:
filename='aux_scripts/1_SAMPLE_QC_HQ_array_vars.sh'

script = '''

set -o pipefail 
set -o errexit

plink \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --maf 0.01 \
    --mac 10 \
    --geno 0.01 \
    --indep-pairwise 1000 50 0.05 \
    --memory 60000 \
    --out array_hq
    
export output_prune_in="array_hq.prune.in"
export output_prune_out="array_hq.prune.out"

mv ${output_prune_in} ${output_prune_out} -t ${OUTPUT_PATH}
'''

with open(filename,'w') as fp:
    fp.write(script)
    
!gsutil cp aux_scripts/1_SAMPLE_QC_HQ_array_vars.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

MACHINE_TYPE="n2-standard-16" 
SCRIPT_NAME="1_SAMPLE_QC_HQ_array_vars.sh"
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}"


dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.2.14" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "${SCRIPT_NAME}" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bimfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input famfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
%%capture
!gsutil cp {my_BUCKET}/dsub/results/HCM_GWAS_V2/SAMPLE_QC/20251030/array_hq.prune.in ../3_output/1_SAMPLE_QC/

In [ ]:
!wc -l ../3_output/1_SAMPLE_QC/array_hq.prune.in

### HQ srWGS ACAF Variants

In [ ]:
filename='aux_scripts/1_SAMPLE_QC_HQ_srWGS_ACAF_vars.sh'

script = '''

set -o pipefail 
set -o errexit

plink \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --maf 0.01 \
    --mac 10 \
    --geno 0.01 \
    --indep-pairwise 1000 50 0.05 \
    --memory 30000 \
    --out srWGS_ACAF_hq_chr"${chr}"
    
export output_prune_in="srWGS_ACAF_hq_chr"${chr}".prune.in"
export output_prune_out="srWGS_ACAF_hq_chr"${chr}".prune.out"

mv ${output_prune_in} ${output_prune_out} -t ${OUTPUT_PATH}
'''

with open(filename,'w') as fp:
    fp.write(script)
    
!gsutil cp aux_scripts/1_SAMPLE_QC_HQ_srWGS_ACAF_vars.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

SCRIPT_NAME="1_SAMPLE_QC_HQ_srWGS_ACAF_vars.sh"
MACHINE_TYPE="n2-standard-8" 
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}"

#Iterate over chromosomes
LOWER=1
UPPER=21

for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
    dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.2.14" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "srWGS_ACAF_HQ_vars_${chromo}" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
    --input bimfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
    --input famfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
    --env chr=${chromo} \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"
done

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
%%capture
!gsutil -m cp {my_bucket}/dsub/results/HCM_GWAS_V2/SAMPLE_QC/20251030/srWGS_ACAF_hq* ../3_output/1_SAMPLE_QC/srWGS_ACAF_hq/

In [ ]:
!find ../3_output/1_SAMPLE_QC/srWGS_ACAF_hq/*.prune.in -type f -print0 | xargs -0 wc -l

In [ ]:
#Now filter each chromosome's plink fileset to these HQ variants and return new plink files
filename='aux_scripts/1_SAMPLE_QC_HQ_srWGS_ACAF_vars_plink1.sh'

script = '''

set -o pipefail 
set -o errexit

plink \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --extract "${srWGS_ACAF_hq_vars}" \
    --memory 30000 \
    --make-bed \
    --out srWGS_ACAF_hq_varfiltered_chr"${chr}"
    
export output_bed="srWGS_ACAF_hq_varfiltered_chr"${chr}".bed"
export output_bim="srWGS_ACAF_hq_varfiltered_chr"${chr}".bim"
export output_fam="srWGS_ACAF_hq_varfiltered_chr"${chr}".fam"

mv ${output_bed} ${output_bim} ${output_fam} -t ${OUTPUT_PATH}
'''

with open(filename,'w') as fp:
    fp.write(script)
    
!gsutil cp aux_scripts/1_SAMPLE_QC_HQ_srWGS_ACAF_vars_plink1.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

SCRIPT_NAME="1_SAMPLE_QC_HQ_srWGS_ACAF_vars_plink1.sh"
MACHINE_TYPE="n2-standard-8" 
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}"

#Iterate over chromosomes
LOWER=1
UPPER=23

for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
    dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.2.14" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "srWGS_ACAF_HQ_vars_${chromo}" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
    --input bimfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
    --input famfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
    --input srWGS_ACAF_hq_vars="${WORKSPACE_BUCKET}/dsub/results/HCM_GWAS_V2/SAMPLE_QC/20251030/srWGS_ACAF_hq_chr${chromo}.prune.in" \
    --env chr=${chromo} \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"
done

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
%%capture
#Download all the plink files after filtering + merge into single file + filter
!gsutil -m cp {my_bucket}/dsub/results/HCM_GWAS_V2/SAMPLE_QC/20251031/srWGS_ACAF_hq* ../3_output/1_SAMPLE_QC/srWGS_ACAF_hq/plink_files/

In [ ]:
#Download all the plink filesets after filtering + merge into single file + filter
!ls ../3_output/1_SAMPLE_QC/srWGS_ACAF_hq/plink_files/*.bed | sed 's/\.bed$//' > ../3_output/1_SAMPLE_QC/srWGS_ACAF_hq/plink_files/bed_prefixes.txt

In [ ]:
#Download all the plink filesets after filtering + merge into single file + filter
!plink --merge-list ../3_output/1_SAMPLE_QC/srWGS_ACAF_hq/plink_files/bed_prefixes.txt --memory 12000 --out ../3_output/1_SAMPLE_QC/srWGS_ACAF_hq/plink_files/merged_srWGS_ACAF_hq

In [ ]:
#Filter for --mind 0.1
!plink --bfile ../3_output/1_SAMPLE_QC/srWGS_ACAF_hq/plink_files/merged_srWGS_ACAF_hq --mind 0.1 --make-just-fam --memory 12000 --out ../3_output/1_SAMPLE_QC/srWGS_ACAF_hq_mind_filtered

## Sample Import



In [ ]:
hcm_gwas_start = pd.read_csv('../1_input/0_cohort_tsvs/HCM_GWAS_START.tsv',sep='\t')
hcm_gwas_start.shape, hcm_gwas_start.columns,sum(hcm_gwas_start['sex_at_birth'].isna()),sum(hcm_gwas_start['race'].isna()),sum(hcm_gwas_start['date_of_birth'].isna())

In [ ]:
!gsutil cp ../1_input/0_cohort_tsvs/HCM_GWAS_START.tsv {my_bucket}/HCM_GWAS_V2/1_input/0_cohort_tsvs/

## QC Procedures

There are 3 main functions of the script which is run on the entire arrays.bed/bim/fam file.
N.B BED file = 181.1 GB for the job instance

1) Output heterozygosity estimates for individuals via plink1.9 --het
2) Output KING-robust kinship estimates and pairs of individuals with >= 0.354 kinship coefficient and output a sample ID list of those after greedy matching to select an individual of each pair via plink2 --king-cutoff 0.354
3) ID genetic sex mismatches by using plink1.9 --check-sex
4) Missingness (> 5% missing hard-called SNPs)

In [ ]:
!gsutil -u $GOOGLE_PROJECT ls gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.*

In [ ]:
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam ../1_input/1_array/
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim ../1_input/1_array/

## Sample QC Run

This actually submits the script via dsub to run and output these PLINK-derived files.

### Heterozygosity Estimation

This is performed on an ancestry-specific basis as per Sealock et al, 2025.

In [ ]:
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv ../1_input/0_cohort_tsvs/

In [ ]:
ancestry_preds = pd.read_csv('../1_input/0_cohort_tsvs/ancestry_preds.tsv', sep='\t').loc[:,['research_id','ancestry_pred']]

In [ ]:
ancestries = ancestry_preds['ancestry_pred'].unique()

def per_ancestry_ID_writer(ancestry_preds, ancestry):
    
    out = ancestry_preds.loc[ancestry_preds['ancestry_pred'] == ancestry,:]
    out = out.assign(FID=0)
    out = out[['FID','research_id']]
    out.to_csv(f'../3_output/0_cohort_tsvs/{ancestry}_ids.tsv', sep='\t', index=False,header=False)
    
    return out
    
perancestry = [per_ancestry_ID_writer(ancestry_preds, x) for x in ancestries]
ancestries

In [ ]:
!gsutil -m cp -r ../3_output/0_cohort_tsvs/ {my_bucket}/HCM_GWAS_V2/3_output/

In [ ]:
filename='aux_scripts/1_SAMPLE_QC_het.sh'

script = '''

set -o pipefail 
set -o errexit

#This outputs the heterozygosity estimates for each individual
plink \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --keep "${ancestry_ids}" \
    --extract "${hq_array_vars}" \
    --het gz \
    --memory 60000 \
    --out array_"${ancestry}"
    
export output_het="array_"${ancestry}".het.gz"

mv ${output_het} -t ${OUTPUT_PATH}
'''

with open(filename,'w') as fp:
    fp.write(script)
    
!gsutil cp aux_scripts/1_SAMPLE_QC_het.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

SCRIPT_NAME="1_SAMPLE_QC_het.sh"
MACHINE_TYPE="n2-standard-16" 
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}" #From above command

ancestries=('afr' 'eur' 'eas' 'amr' 'mid' 'sas')

for ANCESTRY in "${ancestries[@]}"
do
    dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.2.14" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "array_het_${ANCESTRY}" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bimfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input famfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
    --input hq_array_vars="${WORKSPACE_BUCKET}/dsub/results/HCM_GWAS_V2/SAMPLE_QC/20251030/array_hq.prune.in" \
    --input ancestry_ids="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/0_cohort_tsvs/${ANCESTRY}_ids.tsv" \
    --env ancestry=${ANCESTRY} \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"
done

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
%%capture
!gsutil -m cp {my_bucket}/dsub/results/HCM_GWAS_V2/SAMPLE_QC/20251030/*.het.gz ../3_output/1_SAMPLE_QC/array_het/

### Kinship Coefficient Estimation + Filtering

In [ ]:
filename='aux_scripts/1_SAMPLE_QC_kin.sh'

script = '''

set -o pipefail 
set -o errexit

#This outputs the sample IDs after pruning via kinship coefficient
plink2 \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --extract "${hq_array_vars}" \
    --king-cutoff 0.354 \
    --memory 60000 \
    --out array
    
export output_king_in="array.king.cutoff.in.id"
export output_king_out="array.king.cutoff.out.id"

mv ${output_king_in} ${output_king_out} -t ${OUTPUT_PATH}
'''

with open(filename,'w') as fp:
    fp.write(script)
    
!gsutil cp aux_scripts/1_SAMPLE_QC_kin.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

MACHINE_TYPE="n2-standard-16" 
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/1_SAMPLE_QC_kin.sh" #From above command

dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.2.14" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "array_kin" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bimfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input famfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
    --input hq_array_vars="${WORKSPACE_BUCKET}/dsub/results/HCM_GWAS_V2/SAMPLE_QC/20251030/array_hq.prune.in" \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
!gsutil cp {my_bucket}/dsub/results/HCM_GWAS_V2/SAMPLE_QC/20251030/array.king* ../3_output/1_SAMPLE_QC/

### Genetic Sex Mismatches

In [ ]:
filename='aux_scripts/1_SAMPLE_QC_gsex.sh'

script = '''

set -o pipefail 
set -o errexit
    
#This outputs the imputed sex from plink
plink \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --extract "${hq_array_vars}" \
    --split-x b38 no-fail \
    --make-bed \
    --memory 60000 \
    --out split_x
    
plink \
    --bed split_x.bed \
    --bim split_x.bim \
    --fam split_x.fam \
    --check-sex \
    --memory 60000 \
    --out array

export output_sexcheck="array.sexcheck"

mv ${output_sexcheck} -t ${OUTPUT_PATH}
'''

with open(filename,'w') as fp:
    fp.write(script)
    
!gsutil cp aux_scripts/1_SAMPLE_QC_gsex.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

MACHINE_TYPE="n2-standard-16" 
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/1_SAMPLE_QC_gsex.sh" #From above command

dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.2.14" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "array_sexcheck" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bimfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input famfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
    --input hq_array_vars="${WORKSPACE_BUCKET}/dsub/results/HCM_GWAS_V2/SAMPLE_QC/20251030/array_hq.prune.in" \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
%%capture
!gsutil cp {my_bucket}/dsub/results/HCM_GWAS_V2/SAMPLE_QC/20251030/array.sexcheck ../3_output/1_SAMPLE_QC/

### Missingness

In [ ]:
filename='aux_scripts/1_SAMPLE_QC_mind.sh'

script = '''

set -o pipefail 
set -o errexit
    
#This outputs the imputed sex from plink
plink \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --extract "${hq_array_vars}" \
    --mind 0.05 \
    --make-just-fam \
    --memory 60000 \
    --out array_mind_filtered

export output_array_mind_filtered="array_mind_filtered.fam"

mv ${output_array_mind_filtered} -t ${OUTPUT_PATH}
'''

with open(filename,'w') as fp:
    fp.write(script)
    
!gsutil cp aux_scripts/1_SAMPLE_QC_mind.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

MACHINE_TYPE="n2-standard-16" 
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/1_SAMPLE_QC_mind.sh" #From above command

dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.2.14" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "array_mind" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bimfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input famfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
    --input hq_array_vars="${WORKSPACE_BUCKET}/dsub/results/HCM_GWAS_V2/SAMPLE_QC/20251030/array_hq.prune.in" \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
%%capture
!gsutil cp {my_bucket}/dsub/results/HCM_GWAS_V2/SAMPLE_QC/20251030/array_mind_filtered.fam ../3_output/1_SAMPLE_QC/

### Dstat Check

## Array-based Sample Filtering

This actually performs the filtering of the sample IDs to remove those individuals with

1) Excess heterozygosity (outliers beyond median +/- 3 * IQR) on ancestry-specific basis
2) Duplicates or monozygotic twins (PLINK2 KING-robust sample lists)
3) Genetic sex mismatches
4) Missingness (> 5% missing hard-called SNPs)

In [ ]:
#Filter for those who have outlier heterozygosity ratio on ancestry-specific basis
ancestries = ['afr', 'eur', 'eas', 'amr', 'mid', 'sas']
het_results = [pd.read_csv(f'../3_output/1_SAMPLE_QC/array_het/array_{x}.het.gz', sep='\s+') for x in ancestries]

def find_f_outliers(df):
    """
    Calculates the median and 3*IQR for column 'F' in a dataframe
    and returns rows where 'F' is outside this range.
    """
    # Calculate statistics for column 'F'
    F_median = df['F'].median()
    Q1 = df['F'].quantile(0.25)
    Q3 = df['F'].quantile(0.75)
    IQR = Q3 - Q1
    
    # Define the outlier thresholds
    lower_bound = F_median - (3 * IQR)
    upper_bound = F_median + (3 * IQR)
    
    # Create the filter condition
    condition = (df['F'] <= lower_bound) | (df['F'] >= upper_bound)
    
    # Return the filtered DataFrame
    return df[condition]

het_outliers = [find_f_outliers(df) for df in het_results] #Iterate over each on a per-ancestry basis
het_outliers = pd.concat(het_outliers, ignore_index=True).loc[:,'IID'] #Concatenate all ancestry results together
print(het_outliers.shape)

In [ ]:
#Filter for those who are removed due to kinship coefficient
kinship_out = pd.read_csv('../3_output/1_SAMPLE_QC/array.king.cutoff.out.id', sep='\t').loc[:,'IID']
kinship_in = pd.read_csv('../3_output/1_SAMPLE_QC/array.king.cutoff.in.id', sep='\t').loc[:,'IID']
print(kinship_out.shape, kinship_in.shape)

In [ ]:
#Filtering for those who have self-reported sex != genetic sex
#Genetic sex SNPSEX = 1 means Male, 2 means Female (i.e number of X chromosomes)
#Note that the 0 i.e. failed to impute sex are NOT used to filter (i.e. we assume no mismatch for those individuals)

#NOTE THAT THIS ONLY FILTER THOSE WITH SEX FROM THE 313431 EVEN THOUGH IT IS COMPUTED ON ALL ~400K INDIVIDUALS so to reuse, replace hcm_gwas_start!

sexcheck = pd.read_csv('../3_output/1_SAMPLE_QC/array.sexcheck', sep='\s+', header=0).loc[:,['IID', 'SNPSEX']]
sexcheck = sexcheck.rename(columns={'IID':'person_id'})
print(sexcheck['SNPSEX'].value_counts()) #This means that there are 14170 individuals where the array data failed to precisely impute sex
sex_fail = hcm_gwas_start.merge(sexcheck, on='person_id', how='left')
sex_fail = sex_fail.loc[
    ((sex_fail["sex_at_birth"] == "Male") & (sex_fail["SNPSEX"] == 2)) |
    ((sex_fail["sex_at_birth"] == "Female") & (sex_fail["SNPSEX"] == 1)), 'person_id'
]
print(sex_fail.shape) 

In [ ]:
#Filtering for those with excessive missingness in array data
mind_filtered = pd.read_csv('../3_output/1_SAMPLE_QC/array_mind_filtered.fam', sep='\s+', header=None)
mind_fail = hcm_gwas_start.loc[~hcm_gwas_start['person_id'].isin(mind_filtered.iloc[:,1]), 'person_id'] #This filters the HCM_GWAS_START individuals for those passing the mind 0.05 filter
mind_fail.shape

## srWGS-based Sample Filtering

This performs sample filtering based on individuals with known srWGS issues as per 
gs://fc-aou-datasets-controlled/v8/known_issues/

It also performs sample filtering to extract those with >10% missingness in the HQ variants.

In [ ]:
!gsutil -u $GOOGLE_PROJECT ls gs://fc-aou-datasets-controlled/v8/known_issues/

In [ ]:
!gsutil -u $GOOGLE_PROJECT cp -r gs://fc-aou-datasets-controlled/v8/known_issues/ ../1_input/2_srWGS/

In [ ]:
srwgs_issue_ids = [pd.read_csv('../1_input/2_srWGS/known_issues/'+x, sep='\t', header=0) for x in os.listdir('../1_input/2_srWGS/known_issues/')]
srwgs_issue_ids = pd.concat(srwgs_issue_ids, ignore_index=True).loc[:,'research_id']
srwgs_issue_ids.shape

In [ ]:
#This performs filtering for the missingness in HQ variants
srwgs_missing_pass = pd.read_csv('../3_output/1_SAMPLE_QC/srWGS_ACAF_hq_mind_filtered.fam', sep='\s+', header=None).iloc[:,1]
srwgs_missing_pass.shape #414830 = size of the original fam files so no individuals filtered

# Merged Sample Filter

This combines all the filtered samples from the above QC to output the final list of samples which pass sample QC after removing those which don't.

**N.B These sample-level filters are performed on ALL of the individuals with array data + srWGS data to enable ease of resusability. However, the sex_fail checks the imputed sex vs. reported sex of the 313431 individuals (not ALL individuals)**

In [ ]:
sample_qc_array_fail_ids = pd.concat([het_outliers,kinship_out, sex_fail, mind_fail])
sample_qc_srwgs_fail_ids = srwgs_issue_ids
sample_qc_fail_ids = pd.concat([sample_qc_array_fail_ids, srwgs_issue_ids]).unique()
sample_qc_fail_ids.shape #This is of all the individuals (not just the 313,431)

In [ ]:
#Write out to the fail IDs for the total
sample_qc_fail_ids = pd.DataFrame({'IID':sample_qc_fail_ids})
sample_qc_fail_ids['FID'] = 0 
sample_qc_fail_ids = sample_qc_fail_ids.loc[:,['FID','IID']]
sample_qc_fail_ids.to_csv('../3_output/1_SAMPLE_QC/overall_filtered_ids/sample_qc_fail_ids.tsv',sep='\t', index=False)

In [ ]:
#Write out the passed IDs from HCM_GWAS_start
hcm_gwas_sample_qc_pass = hcm_gwas_start.loc[~hcm_gwas_start['person_id'].isin(sample_qc_fail_ids['IID'])]
hcm_gwas_sample_qc_pass_ids = pd.DataFrame({'IID':hcm_gwas_sample_qc_pass['person_id']})
hcm_gwas_sample_qc_pass_ids['FID'] = 0 
hcm_gwas_sample_qc_pass_ids = hcm_gwas_sample_qc_pass_ids.loc[:,['FID','IID']]
hcm_gwas_sample_qc_pass_ids.to_csv('../3_output/1_SAMPLE_QC/overall_filtered_ids/hcm_gwas_sample_qc_pass_ids.tsv',sep='\t', index=False)
hcm_gwas_start.shape, hcm_gwas_sample_qc_pass.shape

This outputs the intersection between this sample-level filtered individuals & each of the genetic ancestry groups as individual .tsv files to 3_output/1_SAMPLE_QC/overall_filtered_ids/

In [ ]:
sample_qc_filtered_ids = pd.read_csv('../3_output/1_SAMPLE_QC/overall_filtered_ids/hcm_gwas_sample_qc_pass_ids.tsv', sep='\t')
sample_qc_filtered_ids.shape

In [ ]:
ancestries = ['afr', 'eur', 'eas', 'amr', 'mid', 'sas']
ancestry_ids = [pd.read_csv(f'../3_output/0_cohort_tsvs/{x}_ids.tsv', sep='\t', header=None) for x in ancestries]
for df in ancestry_ids:
    df.columns = ['FID','IID']

In [ ]:
sample_qc_filtered_ancestry_ids = [pd.merge(sample_qc_filtered_ids, n_df[['FID', 'IID']], on=['FID', 'IID'], how='inner')for n_df in ancestry_ids]
[x.to_csv(f'../3_output/1_SAMPLE_QC/overall_filtered_ids/hcm_gwas_sample_qc_pass_{ancestry}_ids.tsv', sep='\t', index=False) for x,ancestry in zip(sample_qc_filtered_ancestry_ids,ancestries)]

# Misc.

In [ ]:
#Final upload of all the code and the output files from this 1_SAMPLE_QC analysis to GCP Bucket
!gsutil cp 1_SAMPLE_QC.ipynb {my_bucket}/HCM_GWAS_V2/2_scripts/

In [ ]:
!gsutil -m cp -r ../3_output/1_SAMPLE_QC/overall_filtered_ids/ {my_bucket}/HCM_GWAS_V2/3_output/1_SAMPLE_QC/
!gsutil -m cp -r ../3_output/1_SAMPLE_QC/array_het/ {my_bucket}/HCM_GWAS_V2/3_output/1_SAMPLE_QC/

In [ ]:
!gsutil -m cp -r ../3_output/1_SAMPLE_QC/*.fam {my_bucket}/HCM_GWAS_V2/3_output/1_SAMPLE_QC/
!gsutil -m cp -r ../3_output/1_SAMPLE_QC/*.sexcheck {my_bucket}/HCM_GWAS_V2/3_output/1_SAMPLE_QC/
!gsutil -m cp -r ../3_output/1_SAMPLE_QC/*.in   {my_bucket}/HCM_GWAS_V2/3_output/1_SAMPLE_QC/
!gsutil -m cp -r ../3_output/1_SAMPLE_QC/*.id {my_bucket}/HCM_GWAS_V2/3_output/1_SAMPLE_QC/ 

In [ ]:
!gsutil -m cp ../3_output/1_SAMPLE_QC/srWGS_ACAF_hq/*.prune* {my_bucket}/HCM_GWAS_V2/3_output/1_SAMPLE_QC/srWGS_ACAF_hq/